# 03 — BM25 search
Find the project root using the same upward search as notebooks 01 and 02. Import the index and scoring classes; the notebook runs the experiment while reusable logic stays in `bm25_index.py`.

In [1]:
import json
import math
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Find the nearest ancestor containing the data-pipeline directory."""
    for directory in (start, *start.parents):
        if (directory / 'data-pipeline').is_dir():
            return directory
    raise RuntimeError('Run this notebook inside the legal-rag project')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
BM25_DIR = PROJECT_ROOT / 'data-pipeline' / 'notebook_bm25'
PREPROCESSED_DIR = DATA_DIR / 'bm25' / 'preprocessed'
TOKENS_PATH = PREPROCESSED_DIR / 'articles_tokens.jsonl'
CONFIG_PATH = PREPROCESSED_DIR / 'preprocess_config.json'
ARTICLES_PATH = DATA_DIR / 'processed' / 'articles.jsonl'
INDEX_DIR = DATA_DIR / 'bm25' / 'index'
sys.path.insert(0, str(BM25_DIR))
from bm25_index import BM25, BM25Searcher, InvertedIndex, file_sha256, naive_bm25
from text_pipeline import Analyzer, read_jsonl

print('Project root:', PROJECT_ROOT)
print('Preprocessed tokens:', TOKENS_PATH)
print('Index output:', INDEX_DIR)

Project root: /home/panha/ir/project/legal-rag
Preprocessed tokens: /home/panha/ir/project/legal-rag/data-pipeline/data/bm25/preprocessed/articles_tokens.jsonl
Index output: /home/panha/ir/project/legal-rag/data-pipeline/data/bm25/index


Load the saved document analyzer settings before building anything. The same `Analyzer` will process search queries. Changing a stopword, tokenizer, context option, dependency version, or pipeline code causes a clear error instead of mixing incompatible document and query tokens.

In [2]:
analyzer = Analyzer.from_config(CONFIG_PATH)
analyzer.assert_compatible(CONFIG_PATH)
token_records = read_jsonl(TOKENS_PATH)
assert token_records, 'Run notebook 02 first: the token file is empty'
print('Loaded token records:', len(token_records))
print('Distinct canonical article IDs:', len({row['article_id'] for row in token_records}))
print('Tokenizer:', analyzer.tokenizer, '| INCLUDE_CONTEXT:', analyzer.include_context)
print('Sample token record:', token_records[0])
print('Analyzer settings verified against:', CONFIG_PATH)

Loaded token records: 20268
Distinct canonical article IDs: 17540
Tokenizer: khmercut | INCLUDE_CONTEXT: False
Sample token record: {'article_id': '1-1', 'source_record_index': 0, 'tokens': ['មាត្រា', '8', 'មាត្រា', '40', 'មាត្រា', '41', 'មាត្រា', '47', 'មាត្រា', '48', 'មាត្រា', '75', 'មាត្រា', '77', 'មាត្រា', '78', 'មាត្រា', '80', 'ច្បាប់', 'ស្តីពី', 'ចរាចរណ៍', 'ផ្លូវគោក', 'ត្រូវ', 'ប្រកាស', 'ឱ្យ', 'ប្រើ', 'ព្រះរាជ', 'ក្រម', 'លេខ', 'នស', 'រក', 'ម', '0115', '001', 'ចុះ', 'ថ្ងៃទី', '9', 'ខែ', 'មករា', 'ឆ្នាំ', '2015', 'ត្រូវបាន', 'ធ្វើ', 'វិសោធនកម្ម', 'ដូចតទៅ']}
Analyzer settings verified against: /home/panha/ir/project/legal-rag/data-pipeline/data/bm25/preprocessed/preprocess_config.json


Build an inverted index directly from the saved tokens, without splitting or analyzing them again. Each input record gets an internal ID from its source position. Repeated canonical article IDs remain separate records for scoring; the best record wins when search results are collapsed to article IDs.

Here, **N and document frequency count input records**, including repeated canonical IDs. Notebook 02's comparison counted longest-text unique articles, so its comparison statistics describe a different selection. The manifest records this choice and copies the exact analyzer settings. It also fingerprints the saved index and metadata so mixed artifacts are rejected.

In [3]:
manifest = BM25Searcher.build_index(TOKENS_PATH, ARTICLES_PATH, CONFIG_PATH, INDEX_DIR, analyzer=analyzer)
index = InvertedIndex.load(INDEX_DIR)
print('Indexed documents (input records):', index.total_documents)
print('Canonical articles:', manifest['canonical_article_count'])
print('Vocabulary:', len(index.postings))
print('Average record length:', round(index.average_document_length, 4))
print('Files saved:', sorted(path.name for path in INDEX_DIR.iterdir() if path.is_file()))
print('Index counting rule:', manifest['document_unit'])
print('Collapse rule:', manifest['collapse_policy'])
assert index.total_documents == len(token_records)
assert index.tokenization == 'preprocessed_tokens'
print('Saved index reloaded successfully; it contains no k1 or b settings.')

Indexed documents (input records): 20268
Canonical articles: 17540
Vocabulary: 26428
Average record length: 84.4993
Files saved: ['index.json', 'manifest.json', 'records.jsonl']
Index counting rule: input record; duplicates contribute separately to N and df
Collapse rule: highest record score per canonical article_id; never sum duplicates
Saved index reloaded successfully; it contains no k1 or b settings.


BM25 adds a contribution from each analyzed query term:

\[
\mathrm{IDF}(t)=\ln\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right)
\]
\[
\mathrm{score}(d,q)=\sum_{t\in q}\mathrm{IDF}(t)
\frac{tf(t,d)(k_1+1)}{tf(t,d)+k_1(1-b+b\,|d|/\mathrm{avgdl})}.
\]

IDF means *inverse document frequency*: a term found in fewer records gets more weight. Term frequency (`tf`) counts occurrences in a record. `k1` controls how quickly repeated occurrences stop adding much. `b` controls the discount for long records: zero removes it, and one applies full length adjustment. `avgdl` is average record length. Each repeated query term contributes again; a missing term contributes zero.

Use `k1=1.5` and `b=0.75` by default. These are scoring arguments and do not change the index.

In [4]:
K1 = 1.5  # Default term-frequency saturation control.
B = 0.75  # Default strength of record-length adjustment.
TOP_K = 5  # Show five distinct canonical articles per example query.
searcher = BM25Searcher(INDEX_DIR, CONFIG_PATH, analyzer=analyzer, k1=K1, b=B)
print('Loaded verified searcher; k1 =', searcher.scorer.k1, '| b =', searcher.scorer.b)
print('Result fields match notebook 08, plus score and matched_terms.')
print('matched_chunks uses record text; chunk_id identifies the record, and distance is None.')

Loaded verified searcher; k1 = 1.5 | b = 0.75
Result fields match notebook 08, plus score and matched_terms.
matched_chunks uses record text; chunk_id identifies the record, and distance is None.


Check one complete score by hand on a tiny Khmer corpus. Let `d1 = [ច្បាប់, ច្បាប់, កម្មករ]`, `d2 = [ច្បាប់, ពាណិជ្ជកម្ម]`, and `d3 = [កម្មករ]`. Search for `ច្បាប់ កម្មករ`.

There are **N = 3** documents of lengths **3, 2, 1**, so **avgdl = 2**. Both query terms have **df = 2**. With **k1 = 1.5**, **b = 0.75**, each IDF is `ln(1 + 1.5 / 2.5) = ln(1.6) = 0.4700036292457356`.

For `d1`, the length factor is `1 − 0.75 + 0.75 × 3 / 2 = 1.375`.

- `ច្បាប់`, **tf = 2**: `0.4700036292457356 × 2 × 2.5 / (2 + 1.5 × 1.375) = 0.5784660052255207`.
- `កម្មករ`, **tf = 1**: `0.4700036292457356 × 1 × 2.5 / (1 + 1.5 × 1.375) = 0.3836764320373352`.

The total is **0.9621424372628560**. Compare this number with the reusable code below.

In [5]:
HAND_DOCUMENTS = {'d1': ['ច្បាប់', 'ច្បាប់', 'កម្មករ'],
                  'd2': ['ច្បាប់', 'ពាណិជ្ជកម្ម'], 'd3': ['កម្មករ']}
HAND_EXPECTED_SCORE = 0.9621424372628560  # Sum worked out in the markdown above.
SCORE_TOLERANCE = 1e-12  # Allow tiny differences from floating-point arithmetic.
hand_scorer = BM25(InvertedIndex.from_tokens(HAND_DOCUMENTS), k1=K1, b=B)
hand_actual = hand_scorer.score(['ច្បាប់', 'កម្មករ'], 'd1')
print('Hand-computed score:', HAND_EXPECTED_SCORE)
print('Code score:', hand_actual)
print('Absolute difference:', abs(hand_actual - HAND_EXPECTED_SCORE))
assert math.isclose(hand_actual, HAND_EXPECTED_SCORE, rel_tol=SCORE_TOLERANCE, abs_tol=SCORE_TOLERANCE)
print('Hand calculation matches.')

Hand-computed score: 0.962142437262856
Code score: 0.962142437262856
Absolute difference: 0.0
Hand calculation matches.


Choose nine real Khmer searches covering labor, intellectual property, and commerce. The last query writes an article number in Khmer digits. This is a text search: an article number can match a reference inside another article. It is not an exact article-number filter, and the saved analyzer currently excludes context.

In [6]:
EXAMPLE_QUERIES = [
    'ប្រាក់ឈ្នួលអប្បបរមា',
    'ការបញ្ឈប់កិច្ចសន្យាការងារ',
    'ការឈប់សម្រាកមាតុភាព',
    'សិទ្ធិអ្នកនិពន្ធ',
    'ម៉ាកពាណិជ្ជកម្ម',
    'ប៉ាតង់ប្រកាសនីយបត្រតក្កកម្ម',
    'ការចុះបញ្ជីពាណិជ្ជកម្ម',
    'ក្រុមហ៊ុនសហកម្មសិទ្ធិ',
    'មាត្រា១២៣ ការងារ',
]
for query in EXAMPLE_QUERIES:
    print(repr(query), '->', analyzer.analyze(query))
assert analyzer.analyze('មាត្រា១២៣ ការងារ') == analyzer.analyze('មាត្រា123 ការងារ')
print('Khmer and ASCII article-number queries produce identical tokens.')

'ប្រាក់ឈ្នួលអប្បបរមា' -> ['ប្រាក់', 'ឈ្នួល', 'អប្បបរមា']
'ការបញ្ឈប់កិច្ចសន្យាការងារ' -> ['ការបញ្ឈប់', 'កិច្ចសន្យា', 'ការងារ']
'ការឈប់សម្រាកមាតុភាព' -> ['ការឈប់', 'សម្រាក', 'មាតុភាព']
'សិទ្ធិអ្នកនិពន្ធ' -> ['សិទ្ធិ', 'អ្នកនិពន្ធ']
'ម៉ាកពាណិជ្ជកម្ម' -> ['ម៉ាក', 'ពាណិជ្ជកម្ម']
'ប៉ាតង់ប្រកាសនីយបត្រតក្កកម្ម' -> ['ប៉ាតង់', 'ប្រកាស', 'នីយបត្រ', 'តក្កកម្ម']
'ការចុះបញ្ជីពាណិជ្ជកម្ម' -> ['ការចុះបញ្ជី', 'ពាណិជ្ជកម្ម']
'ក្រុមហ៊ុនសហកម្មសិទ្ធិ' -> ['ក្រុមហ៊ុន', 'សហកម្ម', 'សិទ្ធិ']
'មាត្រា១២៣ ការងារ' -> ['មាត្រា', '123', 'ការងារ']
Khmer and ASCII article-number queries produce identical tokens.


Verify the fast scorer with an independent, deliberately slow implementation. The slow version scans every token list, counts term occurrences directly, and computes the formula without using postings or the fast scorer's methods.

Test 20 distinct queries against **every indexed record**, including records with zero scores. The comparison uses both relative and absolute tolerance of `1e-12`. The last query repeats a term to check the repeated-query-term rule.

In [7]:
VALIDATION_QUERIES = EXAMPLE_QUERIES + [
    'កិច្ចសន្យាការងារ', 'ម៉ោងធ្វើការ', 'សុវត្ថិភាពការងារ', 'សហជីពកម្មករ',
    'សំណងកម្មករ', 'ការរំលោភសិទ្ធិអ្នកនិពន្ធ', 'រយៈពេលការពារប៉ាតង់',
    'ការចុះបញ្ជីម៉ាក', 'កិច្ចសន្យាពាណិជ្ជកម្ម', 'ទំនួលខុសត្រូវក្រុមហ៊ុន', 'ច្បាប់ ច្បាប់',
]
MIN_VALIDATION_QUERIES = 20  # Required number of independent full-corpus checks.
assert len(set(VALIDATION_QUERIES)) >= MIN_VALIDATION_QUERIES
documents = {str(row.get('source_record_index', position)): row['tokens']
             for position, row in enumerate(token_records)}
matched_checks = 0
for query in VALIDATION_QUERIES:
    query_tokens = analyzer.analyze(query)
    fast = searcher.scorer.scores(query_tokens)
    slow = naive_bm25(documents, query_tokens, k1=K1, b=B)
    largest_error = max((abs(fast.get(record_id, 0.0) - score) for record_id, score in slow.items()), default=0.0)
    assert set(fast) <= set(slow)
    assert all(math.isclose(fast.get(record_id, 0.0), score,
                           rel_tol=SCORE_TOLERANCE, abs_tol=SCORE_TOLERANCE)
               for record_id, score in slow.items()), query
    matched_checks += 1
    print(f'{matched_checks:02d}. {query}: all {len(slow):,} records match; largest error {largest_error:.3g}', flush=True)
print(f'Naive-versus-fast checks matched: {matched_checks}/{len(VALIDATION_QUERIES)}')

01. ប្រាក់ឈ្នួលអប្បបរមា: all 20,268 records match; largest error 0


02. ការបញ្ឈប់កិច្ចសន្យាការងារ: all 20,268 records match; largest error 0


03. ការឈប់សម្រាកមាតុភាព: all 20,268 records match; largest error 0


04. សិទ្ធិអ្នកនិពន្ធ: all 20,268 records match; largest error 0


05. ម៉ាកពាណិជ្ជកម្ម: all 20,268 records match; largest error 0


06. ប៉ាតង់ប្រកាសនីយបត្រតក្កកម្ម: all 20,268 records match; largest error 0


... (11 outputs left out)

18. កិច្ចសន្យាពាណិជ្ជកម្ម: all 20,268 records match; largest error 0


19. ទំនួលខុសត្រូវក្រុមហ៊ុន: all 20,268 records match; largest error 0


20. ច្បាប់ ច្បាប់: all 20,268 records match; largest error 1.78e-15


Naive-versus-fast checks matched: 20/20


Run the real searches and show the top five canonical articles, with score, ID, a short text preview, and the terms matched by the best record. The result keys match dense notebook 08, with `score` and `matched_terms` added. Higher BM25 scores rank first; scores are useful for ordering results within one query.

In [8]:
PREVIEW_CHARACTERS = 220  # Keep printed article excerpts short enough to scan.
example_results = {}
for query in EXAMPLE_QUERIES:
    results = searcher.search(query, top_k=TOP_K)
    example_results[query] = results
    print()
    print('QUERY:', query)
    print('Returned canonical articles:', len(results))
    assert len({result['article_id'] for result in results}) == len(results)
    for result in results:
        preview = ' '.join(result['full_article'].split())[:PREVIEW_CHARACTERS]
        print(f"{result['rank']}. score={result['score']:.6f} | article={result['article_id']} | {result['title']}")
        print('   Preview:', preview)
        print('   Matched terms:', result['matched_terms'])
print('Result keys:', list(next(result for results in example_results.values() for result in results)))


QUERY: ប្រាក់ឈ្នួលអប្បបរមា
Returned canonical articles: 5
1. score=23.033236 | article=12315-16 | 12315.txt
   Preview: ក្រុមប្រឹក្សាជាតិប្រាក់ឈ្នួលអប្បបរមាមានភារកិច្ចដូចតទៅ៖ - សិក្សាស្រាវជ្រាវតាមបែបវិទ្យាសាស្ត្រលើកិច្ចការទាំងឡាយដែលពាក់ព័ន្ធនឹងប្រាក់ឈ្នួលអប្បបរមា - សម្របសម្រួលនិងសម្រួលលក្ខណៈងាយស្រួលដល់ភាគីពាក់ព័ន្ធក្នុងការសិក្សាស្រាវជ្រាវ
   Matched terms: ['ប្រាក់', 'ឈ្នួល', 'អប្បបរមា']
2. score=22.615023 | article=12315-6 | 12315.txt
   Preview: ការព្រមព្រៀងដោយសារលក្ខខណ្ឌកិច្ចសន្យាដោយផ្ទាល់ណាមួយដែលស្ថិតក្នុងវិសាលភាពអនុវត្តប្រាក់ឈ្នួលអប្បបរមា ហើយដែលតម្រូវឱ្យប្រាក់ឈ្នួលទៅកម្មករនិយោជិតទាបជាងប្រាក់ឈ្នួលអប្បបរមាដែលត្រូវអនុវត្តនោះ ត្រូវទុកជាមោឃៈ។ ការព្រមព្រៀងដោយសារលក្
   Matched terms: ['ប្រាក់', 'ឈ្នួល', 'អប្បបរមា']
3. score=22.156281 | article=12315-11 | 12315.txt
   Preview: រដ្ឋមន្ត្រីក្រសួងទទួលបន្ទុកវិស័យការងារត្រូវចេញប្រកាសកំណត់កាលវិភាគនៃការពិភាក្សាកំណត់ប្រាក់ឈ្នួលអប្បបរមា បន្ទាប់ពីទទួលបានអនុសាសន៍របស់ក្រុមប្រឹក្សាជាតិប្រាក់ឈ្នួលអប្បបរមា។ ក្នុងករណីមានសេចក្តីត្រូវការចាំបាច់ណាមួយ រដ្ឋម


QUERY: ក្រុមហ៊ុនសហកម្មសិទ្ធិ
Returned canonical articles: 5
1. score=20.638738 | article=956-1 | 956.txt
   Preview: ច្បាប់នេះកំណត់ទម្រង់ក្រុមហ៊ុនសហកម្មសិទ្ធិ និងក្រុមហ៊ុនមូលធន ដែលធ្វើពាណិជ្ជកម្មនៅក្នុងព្រះរាជាណាចក្រកម្ពុជា ។ ក្រុមហ៊ុនសហកម្មសិទ្ធិមានក្រុមហ៊ុនសហកម្មសិទ្ធិទូទៅ និងក្រុមហ៊ុនសហកម្មសិទ្ធិមានកម្រិត ។ ក្រុមហ៊ុនមូលធនមានក្រុមហ៊ុ
   Matched terms: ['ក្រុមហ៊ុន', 'សហកម្ម', 'សិទ្ធិ']
2. score=20.599711 | article=956-74 | 956.txt
   Preview: សហកម្មសិទ្ធិករទូទៅមានសិទ្ធិនិងកាតព្វកិច្ចដូចជាសហកម្មសិទ្ធិករនៃក្រុមហ៊ុនសហកម្មសិទ្ធិទូទៅដែរ ៖ សហកម្មសិទ្ធិករទូទៅ ត្រូវរាយការណ៍ពីការគ្រប់គ្រងដល់សហកម្មសិទ្ធិករមានកម្រិតទាំងអស់ ៖ សហកម្មសិទ្ធិករទូទៅមានកាតព្វកិច្ចទទួលខុសត្រូវច
   Matched terms: ['ក្រុមហ៊ុន', 'សហកម្ម', 'សិទ្ធិ']
3. score=20.500096 | article=956-79 | 956.txt
   Preview: សហកម្មសិទ្ធិករមានកម្រិតមិនអាចចូលរួមក្នុងការគ្រប់គ្រងអាជីវកម្មរបស់ក្រុមហ៊ុនសហកម្មសិទ្ធិមានកម្រិតបានទេ ៖ សហកម្មសិទ្ធិករមានកម្រិតអាចធ្វើការពិនិត្យរបាយការណ៍ និងបញ្ជីគណនីរបស់ក្រុមហ៊ុន និងអាចសួរសុំជាគំនិតក្នុងការគ្រប់គ្រងក្រុម

The results mostly look sensible for specific topics. The minimum-wage query returns articles `12315-16`, `12315-6`, and `12315-11`, whose previews discuss minimum wages. Copyright results discuss authors and their rights, and the partnership query returns company partnership articles.

There are visible weaknesses. The maternity-leave query ranks a general leave article first, even though it does not match the maternity token. Commercial-registration results include trademark-registration articles because both mention registration and commerce. BM25 adds evidence from any matching term; it does not require every term or understand the full question.

The article-number query is particularly weak: its first result is an English contents fragment containing `123`, followed by references to old article numbers. Khmer digits were converted correctly, but this is a keyword match, not an exact article-number lookup. These examples should stay in the evaluation set for notebook 04.

Explain a real ranking. For the first result of the minimum-wage query, print each analyzed term's frequency in the best record (`tf`), the number of indexed records containing it (`df`), its IDF, and its score contribution. The contributions must sum to the search result's score.

In [9]:
EXPLAIN_QUERY = EXAMPLE_QUERIES[0]  # Inspect the minimum-wage example.
first_result = example_results[EXPLAIN_QUERY][0]
explanation = searcher.explain(EXPLAIN_QUERY, first_result['article_id'])
explained_total = sum(row['partial_score'] for row in explanation['terms'])
assert math.isclose(explained_total, first_result['score'], rel_tol=SCORE_TOLERANCE, abs_tol=SCORE_TOLERANCE)
print('Contribution total matches ranked score:', explained_total)

Article 12315-16, best record 3996, length 145
term | query frequency | tf | df | IDF | partial score
'ប្រាក់' | 1 | 11 | 2520 | 2.084635330715 | 4.308557624339
'ឈ្នួល' | 1 | 11 | 267 | 4.327728337902 | 8.944618107350
'អប្បបរមា' | 1 | 10 | 164 | 4.813937334033 | 9.780060006154
Total: 23.033235737843
Contribution total matches ranked score: 23.03323573784271


Check three queries that have no useful matches: empty text, only stopwords, and a word absent from the vocabulary. They should return an empty list. Also mix an unknown word with a legal query to show that unknown words contribute zero without hiding the known terms.

In [10]:
UNKNOWN_QUERY = 'missingvocabularyqzx'  # Deliberately invented token absent from this corpus.
assert all(token not in index.postings for token in analyzer.analyze(UNKNOWN_QUERY))
EDGE_QUERIES = {'Empty query': '', 'Only stopwords': 'និង ហើយ ដែល', 'Unknown words': UNKNOWN_QUERY}
for label, query in EDGE_QUERIES.items():
    results = searcher.search(query)
    print(label, '| analyzed tokens:', analyzer.analyze(query), '| results:', results)
    assert results == []
known_results = searcher.search(EXAMPLE_QUERIES[0])
mixed_results = searcher.search(EXAMPLE_QUERIES[0] + ' ' + UNKNOWN_QUERY)
assert mixed_results == known_results
print('Adding an unknown token leaves known-term results unchanged.')

Empty query | analyzed tokens: [] | results: []
Only stopwords | analyzed tokens: [] | results: []
Unknown words | analyzed tokens: ['missingvocabularyqzx'] | results: []


Adding an unknown token leaves known-term results unchanged.


Show clear configuration errors without changing saved files. First, alter only the query analyzer's context setting and show that search refuses it. Then use a temporary copy of the manifest with incompatible settings to prove that the index manifest is checked too.

In [11]:
from tempfile import TemporaryDirectory

incompatible = Analyzer(tokenizer=analyzer.tokenizer, include_context=not analyzer.include_context,
                        syllable_ngram_min=analyzer.syllable_ngram_min,
                        syllable_ngram_max=analyzer.syllable_ngram_max)
try:
    BM25Searcher(INDEX_DIR, CONFIG_PATH, analyzer=incompatible)
except ValueError as error:
    print('Expected query analyzer rejection:', error)
else:
    raise AssertionError('An incompatible query analyzer was accepted')
with TemporaryDirectory() as temporary:
    temporary_index = Path(temporary)
    altered_manifest = json.loads(json.dumps(manifest))
    altered_manifest['analyzer_config']['INCLUDE_CONTEXT'] = not analyzer.include_context
    (temporary_index / 'manifest.json').write_text(json.dumps(altered_manifest), encoding='utf-8')
    try:
        BM25Searcher(temporary_index, CONFIG_PATH, analyzer=analyzer)
    except ValueError as error:
        print('Expected index manifest rejection:', error)
    else:
        raise AssertionError('An incompatible manifest was accepted')

Expected query analyzer rejection: Analyzer configuration mismatch: INCLUDE_CONTEXT
Expected index manifest rejection: Index manifest analyzer settings differ from preprocess_config.json; rebuild notebook 03


Change `k1` and `b` without rebuilding the index. Reload the same saved index with alternate scoring arguments, compare its first result, and confirm the index bytes did not change.

In [12]:
ALTERNATE_K1 = 2.0  # Try slower saturation of repeated terms.
ALTERNATE_B = 0.5  # Try a weaker long-record discount.
original_index_hash = file_sha256(INDEX_DIR / 'index.json')
alternate = BM25Searcher(INDEX_DIR, CONFIG_PATH, k1=ALTERNATE_K1, b=ALTERNATE_B)
for label, active_searcher in [('Default parameters', searcher), ('Alternate parameters', alternate)]:
    result = active_searcher.search(EXPLAIN_QUERY, top_k=1)[0]
    print(label, '| article:', result['article_id'], '| score:', round(result['score'], 6))
assert original_index_hash == file_sha256(INDEX_DIR / 'index.json')
print('Index unchanged; k1 and b are query-time settings.')

Default parameters | article: 12315-16 | score: 23.033236
Alternate parameters | article: 12315-16 | score: 26.785036
Index unchanged; k1 and b are query-time settings.
